In [1]:
#######################
# BANK CHURN PREDICTION
#######################
# import libraries
# pandas for data handling
# numpy for numerical operations
# matplotlib and seaborn for visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML utilities
# split data, hyperparameter tuning
# feature scaling
# chain pre-processing
# performance evaluation
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# machine learning models for comparison
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

########
# DATA
########
# def: load and clean the data
# pd.reads imports the data into a data frame
# convert attribution variable from text to numbers
# convert categorical variables to numeric columns
# drop_first=True avoids multicollinearity
def load_data(path):
    df = pd.read_csv(path)
    
    # conversion
    df['Attrition_Flag'] = df['Attrition_Flag'].map({
        'Existing Customer': 0,
        'Attrited Customer': 1
    })
    
    # convert categorical variables
    categorical_cols = [
        'Gender', 'Education_Level', 'Marital_Status',
        'Income_Category', 'Card_Category'
    ]
    
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    
    return df


###################
# DATA EXPLORATION
###################
def run_eda(df):
    print(df.info())
    print("\nMissing Values:\n", df.isnull().sum())

    # Attrition distribution - shows imbalance
    sns.countplot(x='Attrition_Flag', data=df)
    plt.title("Churn Distribution")
    plt.show()

    # Key correlations with attrition
    corr = df.corr()['Attrition_Flag'].sort_values(ascending=False)
    print("\nTop Correlations:\n", corr.head(10))
    
#############
# SPLIT DATA
#############
# using 80% to model and 20% to test
# define X (these are features) and Y (Attrition)
# set random_state = 42 to ensure that it is reproducible
def split_data(df):
    X = df.drop('Attrition_Flag', axis=1)
    y = df['Attrition_Flag']
    
    return train_test_split(X, y, test_size=0.2, random_state=42)

###########
# TRAINING
###########
# function to train model
def train_model(name, model, param_grid, X_train, y_train, X_test, y_test, scale=True):
    print(f"\n🔹 Training: {name}")
    
    # adds scalar only if needed
    steps = []
    
    if scale:
        steps.append(('scaler', StandardScaler()))
    
    steps.append(('model', model))
    
    # scaling happens before training
    pipeline = Pipeline(steps)
    
    # 5-fold cross validation
    # f1 is best for imbalance
    grid = GridSearchCV(
        pipeline,
        param_grid,
        cv=5,
        scoring='f1'
    )
    
    # train all parameter combinations
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    
    print("Best Params:", grid.best_params_)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    
    return best_model


#########
# MODELS
#########
# all models are defined here
# model, hyperparameters, whether scaling is needed
# class_weight=balanced automatically adjusts for class imbalance, gives more importance to attrition = 1
def get_models():
    return {
        "Logistic Regression": {
            "model": LogisticRegression(max_iter=1000, class_weight='balanced'),
            "params": {
                'model__C': [0.01, 0.1, 1, 10]
            },
            "scale": True
        },
        
        "Decision Tree": {
            "model": DecisionTreeClassifier(),
            "params": {
                'model__max_depth': [3, 5, 10, None],
                'model__min_samples_split': [2, 5, 10]
            },
            "scale": False
        },
        
        "Random Forest": {
            "model": RandomForestClassifier(class_weight='balanced'),
            "params": {
                'model__n_estimators': [100, 200],
                'model__max_depth': [5, 10, None]
            },
            "scale": False
        },
        
        "Gradient Boosting": {
            "model": GradientBoostingClassifier(),
            "params": {
                'model__n_estimators': [100, 200],
                'model__learning_rate': [0.01, 0.1],
                'model__max_depth': [3, 5]
            },
            "scale": False
        },
        
        "AdaBoost": {
            "model": AdaBoostClassifier(),
            "params": {
                'model__n_estimators': [50, 100],
                'model__learning_rate': [0.01, 0.1, 1]
            },
            "scale": False
        },
        
        "SVM": {
            "model": SVC(probability=True),
            "params": {
                'model__C': [0.1, 1, 10],
                'model__kernel': ['rbf', 'linear']
            },
            "scale": True
        },
        
        "KNN": {
            "model": KNeighborsClassifier(),
            "params": {
                'model__n_neighbors': [3, 5, 7],
                'model__weights': ['uniform', 'distance']
            },
            "scale": True
        },
        
        "Naive Bayes": {
            "model": GaussianNB(),
            "params": {
                'model__var_smoothing': [1e-9, 1e-8, 1e-7]
            },
            "scale": True
        },
        
    }


######
# RUN 
######
def run_all_models(models, X_train, y_train, X_test, y_test):
    results = []
    trained_models = {}
    
    # loop through each model configuration
    for name, config in models.items():
        # call training function
        model = train_model(
            name,
            config["model"],
            config["params"],
            X_train, y_train,
            X_test, y_test,
            scale=config["scale"]
        )
        
        trained_models[name] = model
        
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        
        # save accuracy
        results.append((name, acc))
    
    # create ranked comparison table
    results_df = pd.DataFrame(results, columns=['Model', 'Accuracy']) \
                    .sort_values(by='Accuracy', ascending=False)
    
    return trained_models, results_df

#######
# MAIN
#######
# import data
def main():
    df = load_data("C:/Users/eaffu/Documents/Personal_Projects/BankChurners.csv")
    
    # split data    
    X_train, X_test, y_train, y_test = split_data(df)
    
    # load model configurations
    models = get_models()
    # train all models and compare
    trained_models, results_df = run_all_models(
        models, X_train, y_train, X_test, y_test
    )
    
    print("\n Model Comparison:")
    # display final ranking
    print(results_df)

if __name__ == "__main__":
    main()


🔹 Training: Logistic Regression
Best Params: {'model__C': 0.01}
Accuracy: 1.0
ROC-AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1699
           1       1.00      1.00      1.00       327

    accuracy                           1.00      2026
   macro avg       1.00      1.00      1.00      2026
weighted avg       1.00      1.00      1.00      2026


🔹 Training: Decision Tree
Best Params: {'model__max_depth': 3, 'model__min_samples_split': 2}
Accuracy: 1.0
ROC-AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1699
           1       1.00      1.00      1.00       327

    accuracy                           1.00      2026
   macro avg       1.00      1.00      1.00      2026
weighted avg       1.00      1.00      1.00      2026


🔹 Training: Random Forest
Best Params: {'model__max_depth': 5, 'model__n_estimators': 100}
Accuracy: 1.0
ROC-AUC: 1.0
             